# 18vC — Common OOF support, scoring and development-only selection

This stage merges the independently generated baseline/GP and tree
OOF panels, freezes the common candidate support, and applies the
pre-specified development-only selection hierarchy.

The common support contains only date–decision rows for which all
nine candidates have a temporal OOF distribution and for which the
realised development label is admissible under every candidate's
model-specific holdout freeze.

The primary criterion is date-balanced mean CRPS. Mean absolute
error is secondary and the pre-frozen complexity rank is tertiary.
Family winners are retained for the baseline, Gaussian-process and
tree comparisons, alongside the overall development winner.

No holdout or June outcome is loaded.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / ".git").exists():
    raise RuntimeError(
        f"Run this notebook from the repository root, not {ROOT}"
    )

UTC = timezone.utc
STEP = "18vC"

A_DIR = (
    ROOT
    / "data/processed/18vA_residual_model_design_and_features"
)
B1_DIR = (
    ROOT
    / "data/processed/18vB1_baseline_gp_temporal_oof"
)
B2_DIR = (
    ROOT
    / "data/processed/18vB2_tree_temporal_oof"
)

CANDIDATE_PATH = A_DIR / "18vA_candidate_registry.csv"
QUANTILE_PATH = A_DIR / "18vA_quantile_grid.csv"
A_SUMMARY_PATH = A_DIR / "18vA_summary.json"
A_MANIFEST_PATH = A_DIR / "18vA_sha256_manifest.csv"

B1_PREDICTION_PATH = (
    B1_DIR / "18vB1_baseline_gp_oof_predictions.csv"
)
B1_SUMMARY_PATH = B1_DIR / "18vB1_summary.json"
B1_MANIFEST_PATH = B1_DIR / "18vB1_sha256_manifest.csv"

B2_PREDICTION_PATH = (
    B2_DIR / "18vB2_tree_oof_predictions.csv"
)
B2_SUMMARY_PATH = B2_DIR / "18vB2_summary.json"
B2_MANIFEST_PATH = B2_DIR / "18vB2_sha256_manifest.csv"

OUT_DIR = (
    ROOT
    / "data/processed/18vC_common_support_scoring_and_selection"
)
REPORT_DIR = (
    ROOT
    / "reports/18vC_common_support_scoring_and_selection"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_INPUTS = [
    CANDIDATE_PATH,
    QUANTILE_PATH,
    A_SUMMARY_PATH,
    A_MANIFEST_PATH,
    B1_PREDICTION_PATH,
    B1_SUMMARY_PATH,
    B1_MANIFEST_PATH,
    B2_PREDICTION_PATH,
    B2_SUMMARY_PATH,
    B2_MANIFEST_PATH,
]

for path in REQUIRED_INPUTS:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required verified input is missing: {path}"
        )

In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


def parse_bool(
    series: pd.Series,
    *,
    name: str,
) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    parsed = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "yes": True,
                "no": False,
            }
        )
    )

    if parsed.isna().any():
        bad = series.loc[
            parsed.isna()
        ].drop_duplicates().tolist()
        raise ValueError(
            f"Could not parse Boolean column {name}: {bad}"
        )

    return parsed.astype(bool)


def verify_manifest(path: Path) -> None:
    manifest = pd.read_csv(path)
    failures: list[str] = []

    for row in manifest.itertuples(index=False):
        candidate = ROOT / row.path

        if not candidate.is_file():
            failures.append(f"MISSING: {row.path}")
            continue

        if sha256_file(candidate) != row.sha256:
            failures.append(f"HASH: {row.path}")

        if candidate.stat().st_size != int(row.size_bytes):
            failures.append(f"SIZE: {row.path}")

    if failures:
        raise AssertionError(
            f"Manifest verification failed for {path}:\n"
            + "\n".join(failures)
        )


def date_balanced_weights(
    frame: pd.DataFrame,
    date_column: str,
) -> pd.Series:
    n_dates = frame[date_column].nunique()
    rows_per_date = frame.groupby(
        date_column
    )[date_column].transform("size")
    weights = 1.0 / (
        float(n_dates)
        * rows_per_date.astype(float)
    )

    if not np.isclose(
        weights.sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ):
        raise AssertionError(
            "Date-balanced weights do not sum to one."
        )

    return weights


def empirical_crps_from_quantiles(
    quantiles: np.ndarray,
    observations: np.ndarray,
) -> np.ndarray:
    q = np.asarray(quantiles, dtype=float)
    y = np.asarray(observations, dtype=float)

    if q.ndim != 2 or len(q) != len(y):
        raise ValueError(
            "Invalid CRPS input dimensions."
        )
    if not (
        np.diff(q, axis=1) >= -1e-12
    ).all():
        raise ValueError(
            "Predictive quantiles are not monotone."
        )

    m = q.shape[1]
    first = np.mean(
        np.abs(q - y[:, None]),
        axis=1,
    )
    coefficients = (
        2.0 * np.arange(1, m + 1)
        - float(m)
        - 1.0
    )
    half_pairwise = (
        q * coefficients[None, :]
    ).sum(axis=1) / float(m * m)

    return first - half_pairwise


for manifest_path in [
    A_MANIFEST_PATH,
    B1_MANIFEST_PATH,
    B2_MANIFEST_PATH,
]:
    verify_manifest(manifest_path)

summaries = {}
for name, path in [
    ("18vA", A_SUMMARY_PATH),
    ("18vB1", B1_SUMMARY_PATH),
    ("18vB2", B2_SUMMARY_PATH),
]:
    with path.open(encoding="utf-8") as handle:
        summaries[name] = json.load(handle)
    if summaries[name].get("verdict") != "PASS":
        raise AssertionError(
            f"{name} is not a PASS release."
        )

candidates = pd.read_csv(
    CANDIDATE_PATH,
    low_memory=False,
)
quantile_grid = pd.read_csv(
    QUANTILE_PATH,
    low_memory=False,
)
baseline_gp = pd.read_csv(
    B1_PREDICTION_PATH,
    low_memory=False,
)
tree = pd.read_csv(
    B2_PREDICTION_PATH,
    low_memory=False,
)

predictions = pd.concat(
    [baseline_gp, tree],
    ignore_index=True,
)

for frame in [
    baseline_gp,
    tree,
    predictions,
]:
    frame["event_date"] = pd.to_datetime(
        frame["event_date"],
        errors="raise",
    )
    frame[
        "freeze_admissible_for_selection"
    ] = parse_bool(
        frame[
            "freeze_admissible_for_selection"
        ],
        name=(
            "freeze_admissible_for_selection"
        ),
    )

quantile_levels = quantile_grid[
    "quantile_level"
].to_numpy(dtype=float)
residual_columns = quantile_grid[
    "residual_quantile_column"
].tolist()

if len(candidates) != 9:
    raise AssertionError(
        f"Expected nine candidates, found {len(candidates)}."
    )
if len(predictions) != 1296:
    raise AssertionError(
        f"Expected 1,296 candidate OOF rows, "
        f"found {len(predictions)}."
    )

candidate_ids = set(
    candidates["candidate_id"]
)
prediction_candidate_ids = set(
    predictions["candidate_id"]
)

if candidate_ids != prediction_candidate_ids:
    raise AssertionError(
        "Candidate registry and OOF panels differ."
    )

print("Verified and merged 18v OOF inputs: PASS")

Verified and merged 18v OOF inputs: PASS


In [3]:
key = ["event_date", "decision_rule"]

if predictions.duplicated(
    ["candidate_id"] + key
).any():
    raise AssertionError(
        "Duplicate candidate-date-rule OOF rows."
    )

counts = predictions.groupby(
    "candidate_id"
).size()
if not counts.eq(144).all():
    raise AssertionError(
        f"Candidate coverage differs:\n{counts}"
    )

presence = (
    predictions.assign(present=True)
    .pivot_table(
        index=key,
        columns="candidate_id",
        values="present",
        aggfunc="all",
        fill_value=False,
    )
)
freeze = predictions.pivot_table(
    index=key,
    columns="candidate_id",
    values="freeze_admissible_for_selection",
    aggfunc="all",
    fill_value=False,
)

common_mask = (
    presence.all(axis=1)
    & freeze.all(axis=1)
)
common_keys = (
    common_mask.loc[common_mask]
    .rename(
        "common_candidate_selection_support"
    )
    .reset_index()
)

if len(common_keys) != 136:
    raise AssertionError(
        f"Expected 136 common support rows, "
        f"found {len(common_keys)}."
    )

common_keys[
    "date_balanced_selection_weight"
] = date_balanced_weights(
    common_keys,
    "event_date",
)

predictions = predictions.merge(
    common_keys,
    on=key,
    how="left",
    validate="many_to_one",
)
predictions[
    "common_candidate_selection_support"
] = predictions[
    "common_candidate_selection_support"
].fillna(False).astype(bool)

common = predictions.loc[
    predictions[
        "common_candidate_selection_support"
    ]
].copy()

if len(common) != 1224:
    raise AssertionError(
        f"Expected 1,224 common scored rows, "
        f"found {len(common)}."
    )

q = common[
    residual_columns
].to_numpy(dtype=float)
y = common["residual_c"].to_numpy(
    dtype=float
)

common["crps_c"] = (
    empirical_crps_from_quantiles(
        q,
        y,
    )
)
common["predicted_residual_mean_c"] = (
    q.mean(axis=1)
)
common["predicted_residual_median_c"] = (
    q[:, 49]
)
common["mean_error_c"] = (
    common["predicted_residual_mean_c"]
    - common["residual_c"]
)
common["absolute_error_c"] = (
    common["mean_error_c"].abs()
)
common["squared_error_c2"] = (
    common["mean_error_c"] ** 2
)
common["median_absolute_error_c"] = (
    common[
        "predicted_residual_median_c"
    ]
    - common["residual_c"]
).abs()
common["hko_q10"] = (
    common["forecast_daily_max_c"]
    + q[:, 9]
)
common["hko_q90"] = (
    common["forecast_daily_max_c"]
    + q[:, 89]
)
common["interval_80_covered"] = (
    common["hko_daily_max_c"].ge(
        common["hko_q10"]
    )
    & common["hko_daily_max_c"].le(
        common["hko_q90"]
    )
)
common["interval_80_width_c"] = (
    common["hko_q90"]
    - common["hko_q10"]
)

score_rows = []

for candidate_id, group in common.groupby(
    "candidate_id",
    sort=False,
):
    weights = group[
        "date_balanced_selection_weight"
    ].to_numpy(dtype=float)
    metadata = candidates.loc[
        candidates["candidate_id"].eq(
            candidate_id
        )
    ].iloc[0]

    score_rows.append(
        {
            "candidate_id": candidate_id,
            "model_family": metadata[
                "model_family"
            ],
            "scope_type": metadata[
                "scope_type"
            ],
            "distribution_type": metadata[
                "distribution_type"
            ],
            "complexity_rank": int(
                metadata["complexity_rank"]
            ),
            "selection_rows": len(group),
            "selection_dates": group[
                "event_date"
            ].nunique(),
            "date_balanced_mean_crps_c": float(
                np.average(
                    group["crps_c"],
                    weights=weights,
                )
            ),
            "date_balanced_mean_absolute_error_c": float(
                np.average(
                    group["absolute_error_c"],
                    weights=weights,
                )
            ),
            "date_balanced_rmse_c": float(
                np.sqrt(
                    np.average(
                        group["squared_error_c2"],
                        weights=weights,
                    )
                )
            ),
            "date_balanced_median_absolute_error_c": float(
                np.average(
                    group[
                        "median_absolute_error_c"
                    ],
                    weights=weights,
                )
            ),
            "date_balanced_interval_80_coverage": float(
                np.average(
                    group[
                        "interval_80_covered"
                    ].astype(float),
                    weights=weights,
                )
            ),
            "date_balanced_interval_80_mean_width_c": float(
                np.average(
                    group[
                        "interval_80_width_c"
                    ],
                    weights=weights,
                )
            ),
        }
    )

scores = pd.DataFrame(score_rows).sort_values(
    [
        "date_balanced_mean_crps_c",
        "date_balanced_mean_absolute_error_c",
        "complexity_rank",
    ],
    kind="mergesort",
).reset_index(drop=True)
scores["overall_rank"] = np.arange(
    1,
    len(scores) + 1,
)

baseline_pool = scores.loc[
    scores["model_family"].isin(
        ["RAW", "BASELINE"]
    )
]
gp_pool = scores.loc[
    scores["model_family"].eq(
        "GAUSSIAN_PROCESS"
    )
]
tree_pool = scores.loc[
    scores["model_family"].eq("TREE")
]

winners = {
    "selected_baseline": baseline_pool.iloc[0][
        "candidate_id"
    ],
    "selected_gaussian_process": gp_pool.iloc[0][
        "candidate_id"
    ],
    "selected_tree": tree_pool.iloc[0][
        "candidate_id"
    ],
    "selected_overall": scores.iloc[0][
        "candidate_id"
    ],
}

for role, candidate_id in winners.items():
    scores[role] = scores[
        "candidate_id"
    ].eq(candidate_id)

if len(scores) != 9:
    raise AssertionError(
        f"Expected nine score rows, found {len(scores)}."
    )
if not all(
    scores[role].sum() == 1
    for role in winners
):
    raise AssertionError(
        "A selection role lacks one winner."
    )

print("Common support scoring and selection: PASS")
print(f"Common rows: {len(common_keys):,}")
display(scores)
print(json.dumps(winners, indent=2))

Common support scoring and selection: PASS
Common rows: 136


/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99261/1365374602.py:71: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ].fillna(False).astype(bool)


,candidate_id,model_family,scope_type,distribution_type,complexity_rank,selection_rows,selection_dates,date_balanced_mean_crps_c,date_balanced_mean_absolute_error_c,date_balanced_rmse_c,date_balanced_median_absolute_error_c,date_balanced_interval_80_coverage,date_balanced_interval_80_mean_width_c,overall_rank,selected_baseline,selected_gaussian_process,selected_tree,selected_overall
0,pooled_empirical_residual,BASELINE,POOLED,EMPIRICAL_QUANTILES,4,136,36,0.711253,1.001912,1.223199,0.993519,0.652778,2.650381,1,True,False,False,True
1,rule_empirical_residual,BASELINE,RULE_SPECIFIC,EMPIRICAL_QUANTILES,5,136,36,0.723767,0.995948,1.224350,0.977431,0.638889,2.431042,2,False,False,False,False
2,gp_matern32_rule,GAUSSIAN_PROCESS,RULE_SPECIFIC,GAUSSIAN_QUANTILES,7,136,36,0.740041,1.021607,1.252387,1.021607,0.611111,2.309567,3,False,True,False,False
3,gp_rbf_rule,GAUSSIAN_PROCESS,RULE_SPECIFIC,GAUSSIAN_QUANTILES,6,136,36,0.743968,1.028708,1.259107,1.028708,0.618056,2.317252,4,False,False,False,False
4,catboost_quantile_pooled,TREE,POOLED,INTERPOLATED_QUANTILES,8,136,36,0.759924,0.964574,1.237288,0.953304,0.423611,1.411981,5,False,False,True,False
5,rule_mean_residual,BASELINE,RULE_SPECIFIC,DEGENERATE,3,136,36,0.996357,0.996357,1.224671,0.996357,0.000000,0.000000,6,False,False,False,False
6,pooled_mean_residual,BASELINE,POOLED,DEGENERATE,2,136,36,1.002211,1.002211,1.223580,1.002211,0.000000,0.000000,7,False,False,False,False
7,catboost_quantile_rule,TREE,RULE_SPECIFIC,INTERPOLATED_QUANTILES,9,136,36,1.035653,1.126208,1.450308,1.122372,0.104167,0.521703,8,False,False,False,False
8,raw_deterministic,RAW,POOLED,DEGENERATE,1,136,36,1.849306,1.849306,2.102297,1.849306,0.000000,0.000000,9,False,False,False,False


{
  "selected_baseline": "pooled_empirical_residual",
  "selected_gaussian_process": "gp_matern32_rule",
  "selected_tree": "catboost_quantile_pooled",
  "selected_overall": "pooled_empirical_residual"
}


In [4]:
check_rows = []

def add_check(
    check: str,
    passed: bool,
    detail: str,
) -> None:
    check_rows.append(
        {
            "check": check,
            "passed": bool(passed),
            "detail": detail,
            "blocking": True,
        }
    )

add_check(
    "candidate_models_9",
    len(candidates) == 9,
    f"candidates={len(candidates)}",
)
add_check(
    "oof_prediction_rows_1296",
    len(predictions) == 1296,
    f"rows={len(predictions)}",
)
add_check(
    "each_candidate_has_144_rows",
    counts.eq(144).all(),
    counts.to_dict().__str__(),
)
add_check(
    "common_selection_rows_136",
    len(common_keys) == 136,
    f"rows={len(common_keys)}",
)
add_check(
    "common_scored_rows_1224",
    len(common) == 1224,
    f"rows={len(common)}",
)
add_check(
    "selection_weights_sum_to_one",
    np.isclose(
        common_keys[
            "date_balanced_selection_weight"
        ].sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ),
    "date-balanced support",
)
add_check(
    "score_rows_9",
    len(scores) == 9,
    f"rows={len(scores)}",
)
add_check(
    "one_winner_per_role",
    all(
        scores[role].sum() == 1
        for role in winners
    ),
    json.dumps(winners),
)
add_check(
    "development_only",
    predictions["event_date"].le(
        pd.Timestamp("2026-05-21")
    ).all(),
    "no holdout or external outcome",
)
add_check(
    "market_information_absent",
    "p_market" not in predictions.columns,
    "weather residual selection only",
)
add_check(
    "common_support_pending_resolved",
    True,
    "common candidate OOF support frozen",
)

integrity = pd.DataFrame(check_rows)
if not integrity["passed"].all():
    raise AssertionError(
        "18vC blocking checks failed:\n"
        + integrity.loc[
            ~integrity["passed"]
        ].to_string(index=False)
    )

issues = pd.DataFrame(
    columns=[
        "issue_level",
        "issue_code",
        "candidate_id",
        "event_date",
        "decision_rule",
        "detail",
        "blocking",
    ]
)

print("18vC integrity checks: PASS")

18vC integrity checks: PASS


In [5]:
outputs = {
    "all_oof": predictions,
    "common_keys": common_keys,
    "common_scored": common,
    "scores": scores,
    "integrity": integrity,
    "issues": issues,
}

paths = {
    "all_oof": (
        OUT_DIR / "18vC_all_candidate_oof_predictions.csv"
    ),
    "common_keys": (
        OUT_DIR
        / "18vC_common_candidate_selection_support.csv"
    ),
    "common_scored": (
        OUT_DIR
        / "18vC_common_support_scored_predictions.csv"
    ),
    "scores": (
        OUT_DIR / "18vC_candidate_development_scores.csv"
    ),
    "integrity": (
        OUT_DIR / "18vC_integrity_checks.csv"
    ),
    "issues": OUT_DIR / "18vC_issues.csv",
}

for key, frame in outputs.items():
    output = frame.copy()

    for column in output.columns:
        if "date" in column.lower():
            if pd.api.types.is_datetime64_any_dtype(
                output[column]
            ):
                output[column] = output[
                    column
                ].dt.strftime("%Y-%m-%d")

        if (
            "freeze" in column.lower()
            or "cutoff" in column.lower()
            or column.lower().endswith("_utc")
            or "available" in column.lower()
        ):
            output[column] = output[
                column
            ].astype("string")

    output.to_csv(
        paths[key],
        index=False,
    )

selection = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "selection_support_rows": int(
        len(common_keys)
    ),
    "selection_support_dates": int(
        common_keys["event_date"].nunique()
    ),
    "selection_hierarchy": {
        "primary": (
            "date_balanced_mean_crps_c"
        ),
        "secondary": (
            "date_balanced_mean_absolute_error_c"
        ),
        "tertiary": "complexity_rank",
    },
    **winners,
    "holdout_used": False,
    "external_test_used": False,
    "common_candidate_oof_support_intersection_pending": False,
}

selection_path = (
    OUT_DIR / "18vC_selected_candidates.json"
)
selection_path.write_text(
    json.dumps(
        selection,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

source_inventory = pd.DataFrame(
    [
        {
            "input_role": "18vA_candidate_registry",
            "path": str(
                CANDIDATE_PATH.relative_to(ROOT)
            ),
            "rows": len(candidates),
            "sha256": sha256_file(
                CANDIDATE_PATH
            ),
        },
        {
            "input_role": "18vA_quantile_grid",
            "path": str(
                QUANTILE_PATH.relative_to(ROOT)
            ),
            "rows": len(quantile_grid),
            "sha256": sha256_file(
                QUANTILE_PATH
            ),
        },
        {
            "input_role": "18vB1_oof_predictions",
            "path": str(
                B1_PREDICTION_PATH.relative_to(ROOT)
            ),
            "rows": len(baseline_gp),
            "sha256": sha256_file(
                B1_PREDICTION_PATH
            ),
        },
        {
            "input_role": "18vB2_oof_predictions",
            "path": str(
                B2_PREDICTION_PATH.relative_to(ROOT)
            ),
            "rows": len(tree),
            "sha256": sha256_file(
                B2_PREDICTION_PATH
            ),
        },
    ]
)
source_inventory_path = (
    OUT_DIR / "18vC_source_inventory.csv"
)
source_inventory.to_csv(
    source_inventory_path,
    index=False,
)

summary = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": "PASS",
    "candidate_models": int(len(candidates)),
    "oof_prediction_rows": int(
        len(predictions)
    ),
    "rows_per_candidate": 144,
    "common_candidate_selection_rows": int(
        len(common_keys)
    ),
    "common_candidate_selection_dates": int(
        common_keys["event_date"].nunique()
    ),
    "common_support_scored_rows": int(
        len(common)
    ),
    "selected_baseline": winners[
        "selected_baseline"
    ],
    "selected_gaussian_process": winners[
        "selected_gaussian_process"
    ],
    "selected_tree": winners[
        "selected_tree"
    ],
    "selected_overall": winners[
        "selected_overall"
    ],
    "holdout_used": False,
    "external_test_used": False,
    "market_information_used": False,
    "common_candidate_oof_support_intersection_pending": False,
    "issue_rows": 0,
    "integrity_checks_passed": int(
        integrity["passed"].sum()
    ),
    "integrity_checks_total": int(
        len(integrity)
    ),
}

summary_path = OUT_DIR / "18vC_summary.json"
summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

environment = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "revision": "v1",
}
environment_path = (
    OUT_DIR / "18vC_environment.json"
)
environment_path.write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

report_lines = [
    "# 18vC common OOF scoring and selection",
    "",
    "**PASS**",
    "",
    f"- Candidates: {len(candidates):,}",
    f"- Common selection rows: {len(common_keys):,}",
    (
        f"- Common selection dates: "
        f"{common_keys['event_date'].nunique():,}"
    ),
    "",
    "## Candidate scores",
    "",
    (
        "| Rank | Candidate | Family | Scope | "
        "CRPS | MAE | RMSE | 80% coverage |"
    ),
    "|---:|---|---|---|---:|---:|---:|---:|",
]

for row in scores.itertuples(index=False):
    report_lines.append(
        f"| {int(row.overall_rank)} | "
        f"{row.candidate_id} | "
        f"{row.model_family} | "
        f"{row.scope_type} | "
        f"{row.date_balanced_mean_crps_c:.6f} | "
        f"{row.date_balanced_mean_absolute_error_c:.6f} | "
        f"{row.date_balanced_rmse_c:.6f} | "
        f"{row.date_balanced_interval_80_coverage:.6f} |"
    )

report_lines.extend(
    [
        "",
        "## Development-only winners",
        "",
        (
            f"- Baseline: "
            f"{winners['selected_baseline']}"
        ),
        (
            f"- Gaussian process: "
            f"{winners['selected_gaussian_process']}"
        ),
        (
            f"- Tree: {winners['selected_tree']}"
        ),
        (
            f"- Overall: "
            f"{winners['selected_overall']}"
        ),
        "",
        (
            "The common-candidate OOF support is now frozen. "
            "No holdout or external outcome was used."
        ),
    ]
)

report_path = (
    REPORT_DIR
    / "18vC_common_support_scoring_and_selection_report.md"
)
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

manifest_rows = []
for root in [OUT_DIR, REPORT_DIR]:
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "18vC_sha256_manifest.csv":
            continue

        manifest_rows.append(
            {
                "path": str(path.relative_to(ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

manifest_path = (
    OUT_DIR / "18vC_sha256_manifest.csv"
)
pd.DataFrame(manifest_rows).to_csv(
    manifest_path,
    index=False,
)

print(json.dumps(summary, indent=2))
print("18vC selection release: PASS")

{
  "step": "18vC",
  "generated_at_utc": "2026-07-21T22:43:09.332265+00:00",
  "verdict": "PASS",
  "candidate_models": 9,
  "oof_prediction_rows": 1296,
  "rows_per_candidate": 144,
  "common_candidate_selection_rows": 136,
  "common_candidate_selection_dates": 36,
  "common_support_scored_rows": 1224,
  "selected_baseline": "pooled_empirical_residual",
  "selected_gaussian_process": "gp_matern32_rule",
  "selected_tree": "catboost_quantile_pooled",
  "selected_overall": "pooled_empirical_residual",
  "holdout_used": false,
  "external_test_used": false,
  "market_information_used": false,
  "common_candidate_oof_support_intersection_pending": false,
  "issue_rows": 0,
  "integrity_checks_passed": 11,
  "integrity_checks_total": 11
}
18vC selection release: PASS
